# 🏙️ Smart City Digital Twin — Model Training in Google Colab

This notebook trains Machine Learning models on the **Smart City Digital Twin Ecosystem Dataset (UrbanPulse)** and exports artifacts compatible with **Decisera**.

### Pipeline Workflow:
1. **Download Dataset** via `kagglehub` from Kaggle.
2. **Data Exploration & Ingestion** across relational tables (`power_grid`, `weather`, `traffic`, `districts`).
3. **Feature Engineering & Cleaning** (handling missing sensor data, temporal cyclical encoding, spatial joins).
4. **Model Training & Comparison** (XGBoost, LightGBM, Random Forest).
5. **Evaluation & SHAP Explainability**.
6. **Export Artifacts** for Decisera platform.

In [ ]:
# Step 1: Install required dependencies
!pip install -q kagglehub xgboost lightgbm scikit-learn shap pandas numpy matplotlib seaborn joblib

In [ ]:
# Step 2: Download dataset from Kaggle via kagglehub
import kagglehub
import os
from pathlib import Path

path = kagglehub.dataset_download("razanihababdellatif/smart-city-digital-twin-ecosystem-dataset")
print(f"Dataset downloaded to: {path}")

# List available tables
files = list(Path(path).glob("*.csv"))
print("\nAvailable CSV tables:")
for f in files:
    print(f" - {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")

In [ ]:
# Step 3: Load and inspect relational tables
import pandas as pd
import numpy as np

data_dir = Path(path)

# Load primary sensor streams and metadata
df_power = pd.read_csv(data_dir / "power_grid.csv") if (data_dir / "power_grid.csv").exists() else None
df_weather = pd.read_csv(data_dir / "weather.csv") if (data_dir / "weather.csv").exists() else None
df_traffic = pd.read_csv(data_dir / "traffic.csv") if (data_dir / "traffic.csv").exists() else None
df_districts = pd.read_csv(data_dir / "districts.csv") if (data_dir / "districts.csv").exists() else None

print("Power Grid shape:", df_power.shape if df_power is not None else "N/A")
print("Weather shape:", df_weather.shape if df_weather is not None else "N/A")
print("Traffic shape:", df_traffic.shape if df_traffic is not None else "N/A")
print("Districts shape:", df_districts.shape if df_districts is not None else "N/A")

In [ ]:
# Step 4: Multi-table Merging and Feature Engineering
# Let us build a unified dataset for Smart City Grid Load / Congestion forecasting

base_df = df_power.copy() if df_power is not None else df_traffic.copy()

if "timestamp" in base_df.columns:
    base_df["timestamp"] = pd.to_datetime(base_df["timestamp"])
    base_df["hour"] = base_df["timestamp"].dt.hour
    base_df["dayofweek"] = base_df["timestamp"].dt.dayofweek
    base_df["month"] = base_df["timestamp"].dt.month
    base_df["is_weekend"] = base_df["dayofweek"].isin([5, 6]).astype(int)

if df_weather is not None and "timestamp" in df_weather.columns and "district_id" in df_weather.columns:
    df_weather["timestamp"] = pd.to_datetime(df_weather["timestamp"])
    base_df = base_df.merge(df_weather, on=["timestamp", "district_id"], how="left", suffixes=("", "_weather"))

if df_districts is not None and "district_id" in df_districts.columns:
    base_df = base_df.merge(df_districts, on="district_id", how="left")

print(f"Merged master dataset shape: {base_df.shape}")
base_df.head(5)

In [ ]:
# Step 5: Data Cleaning & Preprocessing (Handling IoT Sensor Missing Values)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Drop timestamp columns for standard tabular ML training
feature_df = base_df.drop(columns=["timestamp"], errors="ignore").copy()

# Identify target column (e.g. grid load, demand, or traffic congestion)
numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
candidate_targets = [col for col in numeric_cols if any(k in col.lower() for k in ["load", "demand", "power", "speed", "congestion", "aqi", "pm2_5"])]
target_col = candidate_targets[0] if candidate_targets else numeric_cols[0]
print(f"Selected Target Variable: {target_col}")

# Separate features (X) and target (y)
X = feature_df.drop(columns=[target_col])
y = feature_df[target_col]

# Impute missing values with column median for numerical, mode for categorical
for col in X.columns:
    if X[col].dtype in ["float64", "int64"]:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].astype(str).fillna(X[col].mode()[0] if not X[col].mode().empty else "Unknown")

# One-hot encode remaining categorical columns
X = pd.get_dummies(X, drop_first=True)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train features shape: {X_train.shape}, Test features shape: {X_test.shape}")

In [ ]:
# Step 6: Train & Compare ML Models (AutoML Benchmark)
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    "LightGBM": lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, random_state=42, verbose=-1),
    "XGBoost": xgb.XGBRegressor(n_estimators=150, learning_rate=0.05, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    results[name] = {"R2 Score": r2, "RMSE": rmse, "MAE": mae}
    trained_models[name] = model

results_df = pd.DataFrame(results).T.sort_values(by="R2 Score", ascending=False)
print("\n--- Model Performance Benchmark ---")
display(results_df)

In [ ]:
# Step 7: Explainability with SHAP (Feature Importance)
import shap
import matplotlib.pyplot as plt

best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]
print(f"Computing SHAP values for best model: {best_model_name}")

sample_X = X_test.sample(min(500, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(sample_X)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_X, plot_type="bar", show=True)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_X, show=True)

In [ ]:
# Step 8: Save Model Artifact & Preprocessed Dataset for Decisera
import joblib

os.makedirs("decisera_artifacts", exist_ok=True)

# Save best model
model_path = f"decisera_artifacts/smart_city_{best_model_name.lower()}_model.joblib"
joblib.dump(best_model, model_path)

# Save sample preprocessed dataset to upload to Decisera UI
sample_dataset_path = "decisera_artifacts/smart_city_processed_sample.csv"
base_df.sample(min(10000, len(base_df)), random_state=42).to_csv(sample_dataset_path, index=False)

print(f"✅ Model saved to: {model_path}")
print(f"✅ Processed sample dataset saved to: {sample_dataset_path}")
print("\nYou can now download these files from Colab and import them directly into Decisera!")